# **Install and Import Libraries**

In [1]:
pip install kagglehub


In [2]:
import os
import kagglehub
import pandas as pd

# **Download and Load Kaggle Dataset**

In [3]:
# Download the Olist Brazilian E-commerce dataset and get its local folder path
path = kagglehub.dataset_download('olistbr/brazilian-ecommerce')

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.


In [4]:
# List every file inside that folder so we know what we're working with
for root, dirs, files in os.walk(path):
    for name in files:
        print(os.path.join(root, name))

/kaggle/input/brazilian-ecommerce/olist_customers_dataset.csv
/kaggle/input/brazilian-ecommerce/olist_sellers_dataset.csv
/kaggle/input/brazilian-ecommerce/olist_order_reviews_dataset.csv
/kaggle/input/brazilian-ecommerce/olist_order_items_dataset.csv
/kaggle/input/brazilian-ecommerce/olist_products_dataset.csv
/kaggle/input/brazilian-ecommerce/olist_geolocation_dataset.csv
/kaggle/input/brazilian-ecommerce/product_category_name_translation.csv
/kaggle/input/brazilian-ecommerce/olist_orders_dataset.csv
/kaggle/input/brazilian-ecommerce/olist_order_payments_dataset.csv


# **Data Exploration on the 4 Datasets**

## EDA Olist Customers Dataset

In [5]:
# Build the full file path to the customers CSV so pandas can read it
customers_file_path = os.path.join(path, 'olist_customers_dataset.csv')

In [6]:
# Load the customers data into a DataFrame
customers_df = pd.read_csv(customers_file_path)

In [7]:
# Check the columns, data types, and non-null counts
display(customers_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


None

In [8]:
# Summary statistics for the numeric columns
display(customers_df.describe())

,customer_zip_code_prefix
count,99441.000000
mean,35137.474583
std,29797.938996
min,1003.000000
25%,11347.000000
50%,24416.000000
75%,58900.000000
max,99990.000000


In [9]:
# Count missing values per column
display(customers_df.isnull().sum())

,0
customer_id,0
customer_unique_id,0
customer_zip_code_prefix,0
customer_city,0
customer_state,0


In [10]:
# Preview the first 5 rows of the customers data, before any cleaning
display(customers_df.head(5))

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


This table has no missing values at all, every one of the five columns is fully populated. That's a good sign, but a clean `isnull().sum()` only catches empty cells. It won't catch a wrong data type, a duplicate hiding under a different ID, or messy text formatting. The checks below look for exactly those kinds of issues.

### Check customer_id and customer_unique_id

In [11]:
# Customer_id is generated per ORDER, not per person. Customer_unique_id is the real customer identifier
# Cohort/retention analysis must use customer_unique_id.
print("Unique customer_id:", customers_df['customer_id'].nunique())
print("Unique customer_unique_id:", customers_df['customer_unique_id'].nunique())

Unique customer_id: 99441
Unique customer_unique_id: 96096


There are 99,441 unique `customer_id`s but only 96,096 unique `customer_unique_id`s. That gap of about 3,345 confirms that `customer_id` is really an order level ID, a returning customer gets a new `customer_id` every time they order, while their `customer_unique_id` stays the same. For any retention or repeat purchase analysis, `customer_unique_id` is the column to group by.

### Check Duplicate

In [12]:
# Tells how many repeat customers exist (unique_id appearing more than once), sanity check before cohort building
print("Full row duplicates:", customers_df.duplicated().sum())
print("Duplicate customer_unique_id:", customers_df.duplicated(subset='customer_unique_id').sum())

Full row duplicates: 0
Duplicate customer_unique_id: 3345


Zero full row duplicates, so there's no accidental double loading of the file. The 3,345 duplicate `customer_unique_id` values are expected, they're the same 3,345 repeat customers we just found above, each with more than one `customer_id`. Nothing to fix here, this confirms the data lines up the way we expect.

### Check Zip Code

In [13]:
# Zip code stored as int64, check leading zero loss (zip code length distribution)
# Brazilian zip prefixes can start with 0, as int64 the zero is already dropped, which breaks joins later
zip_len = customers_df['customer_zip_code_prefix'].astype(str).str.len()
zip_len.value_counts()

,count
customer_zip_code_prefix,
5,75446
4,23995


23,995 out of 99,441 zip codes, about 24 percent, are 4 digits instead of 5. That's not 24 percent of customers having a broken zip code, it's 24 percent that lost a leading zero when the column was read in as a number. A zip prefix like `03001` gets stored as the number `3001`. This gets fixed in the cleaning section by converting the column to a zero padded, 5 digit text string.

### Check Customer State

In [14]:
# customer_state, validate against 27 official BR codes
# Typos/unexpected codes won't show as missing, but will break state-level aggregation
print("Unique states:", customers_df['customer_state'].nunique())
print(sorted(customers_df['customer_state'].unique()))

Unique states: 27
['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO']


All 27 official Brazilian state codes are present, and there's nothing else in the list. No typos, no unexpected values. This column is clean as is and needs no fixing.

### Check Customer City

In [15]:
# "sao paulo" vs "São Paulo" vs "SÃO PAULO" fragment city-level grouping if not normalized
print("Unique cities (raw):", customers_df['customer_city'].nunique(), '\n')
print(customers_df['customer_city'].sample(10, random_state=1))

Unique cities (raw): 4119 

90678       nova friburgo
53288        major vieira
23486            sorocaba
50028              japeri
60525           sao paulo
83885    feira de santana
542           sao goncalo
78310      rio de janeiro
13580      rondon do para
98420            curitiba
Name: customer_city, dtype: object


4,119 unique city names, but the sample above shows they're all lowercase, with no accent marks either. That's not an error, it's just an inconsistent style. Left as is, "sao paulo" and "Sao Paulo" would be treated as two different cities in any city level grouping or chart. This gets standardized to title case in the cleaning section.

**Summary of our customer data observations:**
- **No missing values:** All five columns are fully populated.
- **No duplicate rows:** Zero exact duplicates found.
- **`customer_id` versus `customer_unique_id`:** 99,441 unique `customer_id`s but only 96,096 unique `customer_unique_id`s, confirming `customer_id` is per order, not per person. `customer_unique_id` is the correct key for retention analysis.
- **Zip codes lost their leading zero:** About 24 percent of zip codes were stored as 4 digit numbers instead of 5 digit codes, needs a string conversion.
- **States are clean:** All 27 valid Brazilian state codes, no typos.
- **City names are inconsistent:** All lowercase, no accents, needs standardizing to avoid splitting the same city into multiple groups.

## EDA Olist Orders Dataset

In [16]:
# Build the full file path to the orders CSV so pandas can read it
orders_file_path = os.path.join(path, 'olist_orders_dataset.csv')

In [17]:
# Load the orders data into a DataFrame
orders_df = pd.read_csv(orders_file_path)

In [18]:
# Check the columns, data types, and non-null counts
display(orders_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


None

In [19]:
# Summary statistics for the numeric columns
display(orders_df.describe())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,66dea50a8b16d9b4dee7af250b4be1a5,edb027a75a1449115f6b43211ae02a24,delivered,2018-08-02 12:06:07,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-14 20:02:44,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522


In [20]:
# Count missing values per column
display(orders_df.isnull().sum())

,0
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0


In [21]:
# Preview the first 5 rows of the orders data, before any cleaning
display(orders_df.head(5))

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


Unlike the customers table, this one does have missing values, `order_approved_at` (160 missing), `order_delivered_carrier_date` (1,783 missing), and `order_delivered_customer_date` (2,965 missing). Before deciding whether that's a problem, we need to know whether those gaps are random or explained by something else, like an order that was never delivered. The checks below dig into that, along with converting the date columns and looking for dates that are logically out of order.

### Convert Date Columns to Datetime

In [22]:
# The five date columns come in as plain text right now
# We convert them to real datetime values here, before the checks below,
# so the date comparisons that follow (like 'delivered before approved') are
# actually comparing dates, not comparing text that happens to sort the same way
date_cols = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
             'order_delivered_customer_date','order_estimated_delivery_date']

In [23]:
orders_df[date_cols] = orders_df[date_cols].apply(pd.to_datetime)

In [24]:
# Confirm the conversion worked, dtype should now show datetime64 for these columns
display(orders_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


None

### Check Order Status

In [25]:
# Explains why dates are missing (canceled orders never get delivered)
print(orders_df['order_status'].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [26]:
# Confirms missingness is logical, not an error
print(orders_df[orders_df['order_status'] != 'delivered']['order_delivered_customer_date'].isnull().sum(), "non-delivered rows missing delivery date")

2957 non-delivered rows missing delivery date


In [27]:
# Even among delivered orders, check whether any are still missing a delivery date, this should be 0 or close to it
print(orders_df[orders_df['order_status'] == 'delivered']['order_delivered_customer_date'].isnull().sum())

8


The `order_status` breakdown explains most of the missing dates: 96,478 orders are `delivered`, and the rest are spread across `shipped`, `canceled`, `unavailable`, `invoiced`, `processing`, `created`, and `approved`, none of which would have a delivery date yet. That lines up with the 2,957 non-delivered rows missing a delivery date we found. The one thing that doesn't fully line up: 8 orders are marked `delivered` but still have no delivery date. That's a small inconsistency, worth excluding from delivery time calculations later, but not worth guessing a date for.

### Check Impossible Date Order

In [28]:
# Check for impossible date order: approved before purchase (logical error, not visible as missing/wrong dtype)
print((orders_df['order_approved_at'] < orders_df['order_purchase_timestamp']).sum(), "rows approved before purchase")

0 rows approved before purchase


In [29]:
# Check for impossible date order: delivered to carrier before approved
print((orders_df['order_delivered_carrier_date'] < orders_df['order_approved_at']).sum(), "rows delivered to carrier before approval")

1359 rows delivered to carrier before approval


In [30]:
# Check for impossible date order: delivered to customer before delivered to carrier
print((orders_df['order_delivered_customer_date'] < orders_df['order_delivered_carrier_date']).sum(), "rows customer delivery before carrier delivery")

23 rows customer delivery before carrier delivery


No orders were approved before they were purchased, that's a good sign. But two real issues showed up: 1,359 orders were marked as handed to the carrier before they were even approved, and 23 orders were marked as delivered to the customer before the carrier had picked them up. Both are logically impossible and most likely small logging errors on Olist's side rather than something we can fix from here. We're keeping these rows rather than dropping them, since we don't know the correct dates, but flagging them as a known limitation for any analysis that needs precise delivery timing.

**Summary of our orders data observations:**
- **Dates converted early:** All five date columns are now real datetime values, not text, so the checks in this section and the cleaning step later on both compare actual dates.
- **Missing dates are explained, not random:** They come from orders that were never delivered (canceled, still shipping, etc), not from broken records.
- **One inconsistency to flag:** 8 orders say `delivered` but have no delivery date, exclude these from delivery time calculations.
- **Two logging issues found:** 1,359 orders show carrier pickup before approval, and 23 orders show customer delivery before carrier pickup. Both are kept in the data but noted as a known limitation.

## EDA Olist Order Payments Dataset

In [31]:
# Build the full file path to the payments CSV so pandas can read it
payments_file_path = os.path.join(path, 'olist_order_payments_dataset.csv')

In [32]:
# Load the payments data into a DataFrame
payments_df = pd.read_csv(payments_file_path)

In [33]:
# Check the columns, data types, and non-null counts
display(payments_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


None

In [34]:
# Summary statistics for the numeric columns
display(payments_df.describe())

,payment_sequential,payment_installments,payment_value
count,103886.000000,103886.000000,103886.000000
mean,1.092679,2.853349,154.100380
std,0.706584,2.687051,217.494064
min,1.000000,0.000000,0.000000
25%,1.000000,1.000000,56.790000
50%,1.000000,1.000000,100.000000
75%,1.000000,4.000000,171.837500
max,29.000000,24.000000,13664.080000


In [35]:
# Count missing values per column
display(payments_df.isnull().sum())

,0
order_id,0
payment_sequential,0
payment_type,0
payment_installments,0
payment_value,0


In [36]:
# Preview the first 5 rows of the payments data, before any cleaning
display(payments_df.head(5))

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


This table doesn't have any obvious missing values. However, when we look closely at the payment details, we find some unusual things that a simple check for empty cells wouldn't catch:

*   `payment_installments` (how many payments were made) sometimes shows 0, which doesn't make sense because even a single payment should count as 1 installment.
*   `payment_value` (the amount paid) sometimes shows 0.00, which is odd for an actual purchase.

The next steps will investigate these strange entries before we decide how to handle them.

### Check Payment Installments

In [37]:
# Check rows where payment_installments is 0
# Logically odd, need to see what payment_type they belong to before deciding anything
display(payments_df[payments_df['payment_installments'] == 0])

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


### Check Payment Value

In [38]:
# Check rows where payment_value is 0
# Could be legit (e.g. voucher fully covering an order) or a broken record
display(payments_df[payments_df['payment_value'] == 0])

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


For the nine rows where `payment_value` was 0, six of them were 'voucher' payments (which might be legitimate if a voucher fully covered the cost). The other three were 'not_defined' payment types, so it's a good idea to check how common this 'not_defined' type is overall.

### Check Payment Type

In [39]:
# Check payment_type categories, look for a 'not_defined' or similar catch-all hiding inside a valid-looking string column (won't show up as NaN)
print(payments_df['payment_type'].value_counts())

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


Only 3 out of 103,886 rows have `not_defined` as their payment type. While this is a tiny number, it's a hidden 'unknown' category that a simple check for empty values wouldn't find, because 'not_defined' is a regular word, not an empty spot.

### Check Order_id

In [40]:
# Confirm multiple rows per order_id are legit multi-payment splits not duplicates (Olist allows splitting one order across payment methods)
display(payments_df['order_id'].value_counts().head())

,count
order_id,
fa65dad1b0e818e3ccc5cb0e39231352,29
ccf804e764ed5650cd8759557269dc13,26
285c2e15bebd4ac83635ccc563dc71f4,22
895ab968e7bb0d5659d16cd74cd1650c,21
fedcd9f7ccdc8cba3a18defedd1a5547,19


In [41]:
# Cross-check: max payment_sequential per order_id, to see how many payments are typically split per order
print(payments_df.groupby('order_id')['payment_sequential'].max().value_counts().head())

payment_sequential
1    96401
2     2458
3      303
4      108
5       52
Name: count, dtype: int64


This shows that having several rows for the same `order_id` is actually correct. It means customers often split their payments, for instance, using a gift voucher for part of the cost and a credit card for the rest. These are not accidental duplicate entries. Most orders (96,401) are paid with a single method, but others use two or more.

### Check Top 5 Highest Payment Value

In [42]:
# Confirm they're legitimate large orders, not data entry errors (e.g. misplaced decimal)
display(payments_df.nlargest(5, 'payment_value'))

,order_id,payment_sequential,payment_type,payment_installments,payment_value
52107,03caa2c082116e1d31e67e9ae3700499,1,credit_card,1,13664.08
34370,736e1922ae60d0d6a89247b851902527,1,boleto,1,7274.88
41419,0812eb902a67711a1cb742b3cdaa65ae,1,credit_card,8,6929.31
49581,fefacc66af859508bf1a7934eab1e97f,1,boleto,1,6922.21
85539,f5136e38d1a14a4dbd87dff67da82701,1,boleto,1,6726.66


## EDA Olist Order Reviews Dataset

In [43]:
# Build the full file path to the reviews CSV so pandas can read it
reviews_file_path = os.path.join(path, 'olist_order_reviews_dataset.csv')

In [44]:
# Load the reviews data into a DataFrame
reviews_df = pd.read_csv(reviews_file_path)

In [45]:
# Check the columns, data types, and non-null counts
display(reviews_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


None

In [46]:
# Summary statistics for the numeric columns
display(reviews_df.describe())

,review_score
count,99224.000000
mean,4.086421
std,1.347579
min,1.000000
25%,4.000000
50%,5.000000
75%,5.000000
max,5.000000


In [47]:
# Count missing values per column
display(reviews_df.isnull().sum())

,0
review_id,0
order_id,0
review_score,0
review_comment_title,87656
review_comment_message,58247
review_creation_date,0
review_answer_timestamp,0


It might seem concerning that `review_comment_title` (with 87,656 missing values) and `review_comment_message` (with 58,247 missing values) have so many gaps. However, this is actually normal. Most customers simply leave a star rating without writing a comment. We'll treat this as genuine customer behavior, not as a data error, and leave these missing values as they are (`NaN`).


In [48]:
# Preview the first 5 rows of the reviews data, before any cleaning
display(reviews_df.head(5))

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


### Check Review Score

In [49]:
# Confirm review_score only contains valid values (1-5) and see the distribution shape for later interpretation
print(reviews_df['review_score'].value_counts())

review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64


The `review_score` data is very clean, containing only values from 1 to 5. The distribution is generally positive, with most reviews giving 4 or 5 stars. However, there's also a noticeable, smaller group of 1-star reviews. This information will be helpful when we interpret the data later.

The next checks will focus on converting data types, ensuring the review scores are valid, and most importantly, verifying that `review_id` and `order_id` are unique when they should be.

### Check Review_id

In [50]:
# Check for duplicate review_id
# Each review should be a unique record, duplicates would need investigating
print(reviews_df['review_id'].duplicated().sum())

814


### Check Order_id

In [51]:
# Check for duplicate order_id
# Unclear if Olist allows multiple reviews per order. Affects how you join this table later
print(reviews_df['order_id'].duplicated().sum())

551


We found 814 duplicate `review_id`s and 551 duplicate `order_id`s. This is a bit puzzling, as a duplicate could mean two very different things: either a simple error in the data export, or it could reveal a real pattern in how reviews or orders are handled. We'll need to investigate these cases more closely before making any decisions.

In [52]:
# Inspect the duplicate order_id rows directly
# Are these true duplicates, or genuinely separate reviews?
reviews_df[reviews_df['order_id'].duplicated(keep=False)].sort_values('order_id').head(10)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
25612,89a02c45c340aeeb1354a24e7d4b2c1e,0035246a40f520710769010f752e7507,5,NaN,NaN,2017-08-29 00:00:00,2017-08-30 01:59:12
22423,2a74b0559eb58fc1ff842ecc999594cb,0035246a40f520710769010f752e7507,5,NaN,Estou acostumada a comprar produtos pelo barat...,2017-08-25 00:00:00,2017-08-29 21:45:57
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,NaN,NaN,2018-02-22 00:00:00,2018-02-23 12:12:30
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,NaN,NaN,2018-03-04 00:00:00,2018-03-05 17:02:00
854,830636803620cdf8b6ffaf1b2f6e92b2,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30 00:00:00,2018-01-02 10:54:06
83224,d8e8c42271c8fb67b9dad95d98c8ff80,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30 00:00:00,2018-01-02 10:54:47
17582,017f0e1ea6386de662cbeba299c59ad1,02355020fd0a40a0d56df9f6ff060413,1,NaN,ja reclamei varias vezes e ate hoje não sei on...,2018-03-29 00:00:00,2018-03-30 03:16:19
89888,0c8e7347f1cdd2aede37371543e3d163,02355020fd0a40a0d56df9f6ff060413,3,NaN,UM DOS PRODUTOS (ENTREGA02) COMPRADOS NESTE PE...,2018-03-21 00:00:00,2018-03-22 01:32:08
55137,61fe4e7d1ae801bbe169eb67b86c6eda,029863af4b968de1e5d6a82782e662f5,4,NaN,NaN,2017-07-19 00:00:00,2017-07-20 12:06:11
37911,04d945e95c788a3aa1ffbee42105637b,029863af4b968de1e5d6a82782e662f5,5,NaN,NaN,2017-07-14 00:00:00,2017-07-17 13:58:06


After looking closely, these actually appear to be genuine, separate reviews. We found different review IDs, sometimes different scores, and different submission dates for the same order. This could mean that Olist asked for feedback again, or that customers updated their original review. Because of this, we can't simply remove these as duplicates. When we combine this data later, we'll need a specific plan to decide which review to keep (we'll generally go with the most recent one for each order).

### Check Review_id

In [53]:
# Inspect the duplicate review_id rows directly
# Are these true duplicates, or something structural?
reviews_df[reviews_df['review_id'].duplicated(keep=False)].sort_values('review_id').head(10)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


We noticed something interesting: the exact same review (with the same score, comments, and time stamps) sometimes shows up for two *different* orders. This likely happens when a single customer checkout is internally split by Olist into multiple orders (for instance, if items come from different sellers), with the customer's one review then attached to each resulting order. When we join reviews to orders, we keep these as is. However, for any metric that calculates overall review sentiment (like an average review score), we'll need to remove these duplicates by `review_id` to avoid counting the same review multiple times.

### Check Review Answer timestamp

In [54]:
# Check for logically impossible dates: review answered before it was even created
print((reviews_df['review_answer_timestamp'] < reviews_df['review_creation_date']).sum())

0


**Summary of our review data observations:**
- **No data type issues:** All date columns have been correctly converted.
- **Missing comments are normal:** The many empty fields for `review_comment_title` and `review_comment_message` are not errors. They reflect real customer behavior where most people leave a star rating without writing a comment. We'll leave these as they are.
- **Clean review scores:** The `review_score` values are perfectly valid, ranging from 1 to 5.
- **551 duplicate `order_id`s:** These were genuinely separate reviews submitted over time for the same order. When cleaning, we decided to keep the *most recent* review for each order.
- **814 duplicate `review_id`s:** As explained above, these are cases where one review is shared across multiple orders. To handle this, we're keeping two versions of the cleaned table: one for order-level analysis (where each order retains its associated review) and another for review-level metrics (where each unique `review_id` is counted only once).
- **No impossible dates:** We didn't find any logical errors where a review was answered before it was created.

# **Data Cleaning**

## Cleaning Olist Customer Dataset

When we looked at the customer data, we noticed a few things that needed attention:

**Zip Codes:** See the zip code finding in EDA above, about 24 percent of zip codes lost their leading zero. Fixing it here by converting the column to a zero padded, 5 digit string.

In [55]:
customers_df['customer_zip_code_prefix'] = customers_df['customer_zip_code_prefix'].astype(str).str.zfill(5)

In [56]:
# Quick check that zip codes are now 5-digit strings with the leading zero preserved
customers_df['customer_zip_code_prefix']

,customer_zip_code_prefix
0,14409
1,09790
2,01151
3,08775
4,13056
...,...
99436,03937
99437,06764
99438,60115
99439,92120


**City Names:** See the city casing finding in EDA above, all lowercase and inconsistent. Converting to title case here so the same city doesn't get split into multiple groups.

In [57]:
# Apply the title case formatting to city names
customers_df['customer_city'] = customers_df['customer_city'].str.strip().str.title()

**Customer IDs:** See the `customer_id` versus `customer_unique_id` finding in EDA above. No fix needed here, we just confirm below that nothing is broken and keep using `customer_unique_id` for anything related to repeat customers.

In [58]:
# Confirm no true duplicate rows exist, and that customer_id is unique per row (no dedup needed here).
print("Exact duplicate rows:", customers_df.duplicated().sum())
print("Duplicate customer_id:", customers_df['customer_id'].duplicated().sum())
print("Duplicate customer_unique_id (this is expected for repeat customers, NOT removed):", customers_df['customer_unique_id'].duplicated().sum())

Exact duplicate rows: 0
Duplicate customer_id: 0
Duplicate customer_unique_id (this is expected for repeat customers, NOT removed): 3345


## Cleaning Olist Orders Dataset

When we looked at the orders data, we found a few important things:

**Date Formats:** Already converted to `datetime` earlier, in the "Convert Date Columns to Datetime" step during EDA. No extra conversion needed here.

**Missing Dates:** See the missing dates finding in EDA above, they're explained by order status, not random gaps. We're not dropping any rows, just adding an `is_delivered` flag below so delivery specific metrics can filter on it later.

In [59]:
# Create the is_delivered flag
orders_df['is_delivered'] = orders_df['order_status'] == 'delivered'

**Known Exceptions:** See the impossible date findings in EDA above (8 delivered orders with no delivery date, 1,359 orders handed to carrier before approval, 23 orders delivered before carrier pickup). We're not fixing these, since we'd only be guessing at the correct dates, just keeping them as a known limitation.

## Cleaning Olist Order Payments Dataset

We found two rows where payment_installments was 0. Both were credit card payments with actual money involved, which doesn't make sense (even a single payment means 1 installment). It's likely just a mistake and should be '1'.

In [60]:
# Fix the two rows where installments was recorded as 0, treat them as a single payment (1 installment)
payments_df.loc[payments_df['payment_installments'] == 0, 'payment_installments'] = 1

In [61]:
# Confirm the fix worked, this should print 0
print("Remaining rows with 0 installments:", (payments_df['payment_installments'] == 0).sum())

Remaining rows with 0 installments: 0


## Cleaning Olist Order Reviews Dataset

When we looked at the customer reviews, here's what we found and how we're handling it:

**Date Formats:** The review_creation_date and review_answer_timestamp columns were stored as text, so we've converted them into proper datetime formats. This allows us to do accurate time-based analysis.

In [62]:
# Convert date columns to datetime first, so date logic checks below work correctly (currently stored as string/object)
reviews_df[['review_creation_date','review_answer_timestamp']] = reviews_df[['review_creation_date','review_answer_timestamp']].apply(pd.to_datetime)

In [63]:
display(reviews_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                99224 non-null  object        
 1   order_id                 99224 non-null  object        
 2   review_score             99224 non-null  int64         
 3   review_comment_title     11568 non-null  object        
 4   review_comment_message   40977 non-null  object        
 5   review_creation_date     99224 non-null  datetime64[ns]
 6   review_answer_timestamp  99224 non-null  datetime64[ns]
dtypes: datetime64[ns](2), int64(1), object(4)
memory usage: 5.3+ MB


None

In [64]:
# Version 1: for joining to orders (order-level analysis), one row per order_id, keep the most recent review
reviews_for_orders = reviews_df.sort_values('review_answer_timestamp').drop_duplicates(subset='order_id', keep='last')

In [65]:
reviews_for_orders.head(5)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
37547,6916ca4502d6d3bfd39818759d55d536,bfbd0f9bdef84302105ad712db648a6c,1,NaN,nao recebi o produto e nem resposta da empresa,2016-10-06 00:00:00,2016-10-07 18:32:28
5503,49f695dffa457eaba90d388a5c37e942,e5215415bb6f76fe3b7cb68103a0d1c0,1,NaN,"PRODUTO NÃO CHEGOU,E JÁ PASSOU O PRAZO DE ENTREGA",2016-10-09 00:00:00,2016-10-11 14:31:29
60439,743d98b1a4782f0646898fc915ef002a,e2144124f98f3bf46939bc5183104041,4,NaN,NaN,2016-10-15 00:00:00,2016-10-16 03:20:17
28075,53752edb26544dd41c1209f582c9c589,b8b9d7046c083150cb5360b83a8ebb51,5,NaN,O pedido foi entregue antes do prazo pr0metido,2016-10-16 01:00:00,2016-10-16 15:45:11
41042,b2d5d8db2a841d27a72e4c06c6212368,9aa3197e4887919fde0307fc23601d7a,4,NaN,Só chegou uma parte do pedido ate agora..,2016-10-15 00:00:00,2016-10-17 21:02:49


In [66]:
# Version 2: for review-level aggregate stats (e.g. average review score), dedupe by review_id to avoid double-counting the same review that is shared across multiple order_ids (814 known cases)
reviews_unique = reviews_df.drop_duplicates(subset='review_id', keep='first')
print("reviews_unique rows:", len(reviews_unique))

reviews_unique rows: 98410


In [67]:
print("reviews_df rows:", len(reviews_df))
print("reviews_for_orders rows:", len(reviews_for_orders))

reviews_df rows: 99224
reviews_for_orders rows: 98673


See the duplicate review finding in EDA above, 814 duplicate `review_id`s shared across multiple orders. That's why we keep two versions here: `reviews_for_orders` for order level joins, and `reviews_unique` for review level stats like the average score.

## Referential Integrity Check (across all four cleaned tables)

Before we load all our cleaned data into PostgreSQL (our database), it's really important to do a **Referential Integrity Check**. Think of this as making sure all the puzzle pieces fit together perfectly across our different tables.

Why do we do this? It helps us catch any problems early on. If the IDs we use to link tables (like `order_id` or `customer_id`) don't match up correctly, we could end up with missing information or strange results later when we try to combine data in our database. This check ensures everything lines up nicely, preventing unexpected data loss or errors when we start analyzing the combined information.

In [68]:
# Confirm every order's customer_id exists in the customers table
print(orders_df['customer_id'].isin(customers_df['customer_id']).value_counts())

customer_id
True    99441
Name: count, dtype: int64


In [69]:
# Confirm every payment's order_id exists in the orders table
print(payments_df['order_id'].isin(orders_df['order_id']).value_counts())

order_id
True    103886
Name: count, dtype: int64


In [70]:
# Confirm every review's order_id (order-level version) exists in the orders table
print(reviews_for_orders['order_id'].isin(orders_df['order_id']).value_counts())

order_id
True    98673
Name: count, dtype: int64


# **Save Cleaned Datasets**

In [71]:
# Create an output folder for the cleaned CSV files if it doesn't already exist
output_dir = 'cleaned_datasets'
os.makedirs(output_dir, exist_ok=True)

In [72]:
# Save each cleaned table to its own CSV file, ready to be loaded into PostgreSQL
customers_df.to_csv(os.path.join(output_dir, 'customers_cleaned.csv'), index=False)
orders_df.to_csv(os.path.join(output_dir, 'orders_cleaned.csv'), index=False)
payments_df.to_csv(os.path.join(output_dir, 'payments_cleaned.csv'), index=False)
reviews_for_orders.to_csv(os.path.join(output_dir, 'reviews_for_orders_cleaned.csv'), index=False)
reviews_unique.to_csv(os.path.join(output_dir, 'reviews_unique_cleaned.csv'), index=False)

print(f"Cleaned datasets saved to the '{output_dir}' directory.")

Cleaned datasets saved to the 'cleaned_datasets' directory.


We've successfully gone through all four of our main tables (`customers`, `orders`, `payments`, and `reviews`). Each table has been thoroughly examined, cleaned up, and double-checked to make sure all the connections between them are solid.

**Here's the key takeaway:** We didn't throw out any rows or guess at any missing information. Every missing value or repeated pattern we found was either a natural part of how the data was recorded (like orders that were never delivered, or customers who didn't leave a comment) or reflected a real business process (like customers splitting their payments or orders being shipped from different places). Everything has been carefully documented so we know exactly what we're working with.